In [5]:
import pandas as pd
import re
from collections import defaultdict
import json

class DrugSynonymGenerator:
    def __init__(self):
        self.synonym_dict = defaultdict(set)
        
    def extract_components(self, drug_name):
        """약제명에서 구성요소 추출"""
        components = {
            'product_name': None,
            'ingredient_kr': None,
            'ingredient_en': None,
            'dosage_form': None,
            'company': None
        }
        
        # 제품명과 성분명 분리
        # 패턴: 제품명(성분명)_용량정보
        pattern = r'^([^(]+?)(?:\d+[밀리그램|mg|그램|g])?(?:\(([^)]+)\))?'
        match = re.match(pattern, drug_name)
        
        if match:
            components['product_name'] = self.clean_product_name(match.group(1))
            if match.group(2):
                components['ingredient_kr'] = match.group(2)
        
        # 제형 추출
        dosage_forms = {
            '정': ['정', '정제', 'tab', 'tablet'],
            '캡슐': ['캡슐', 'cap', 'capsule'],
            '시럽': ['시럽', 'syrup', 'syr'],
            '주': ['주', '주사', 'inj', 'injection'],
            '액': ['액', '점안액', 'solution'],
            '겔': ['겔', 'gel'],
            '서방정': ['서방정', 'SR', 'XR', 'ER'],
            '건조시럽': ['건조시럽', 'dry syrup']
        }
        
        for form_key, form_values in dosage_forms.items():
            if any(form in drug_name for form in form_values):
                components['dosage_form'] = form_key
                break
        
        return components
    
    def clean_product_name(self, name):
        """제품명에서 불필요한 요소 제거"""
        # 숫자, 용량 정보 제거
        patterns_to_remove = [
            r'\d+\.?\d*\s*(밀리그램|mg|그램|g|ml|밀리리터|%)',
            r'\d+:\d+',  # 비율
            r'[0-9]+',    # 남은 숫자
            r'\s+',       # 여백 정리
        ]
        
        clean_name = name
        for pattern in patterns_to_remove:
            clean_name = re.sub(pattern, '', clean_name)
        
        return clean_name.strip()
    
    def process_csv_data(self, df):
        """CSV 데이터에서 유의어 관계 생성"""
        
        for _, row in df.iterrows():
            product_name = row['제품명']
            atc_name = row['ATC코드_명칭']
            eng_ingredient = row['eng_ingredient_from_benefit']
            
            # 제품명 처리
            if pd.notna(product_name):
                components = self.extract_components(product_name)
                
                # 제품명 기반 유의어
                if components['product_name']:
                    base_name = components['product_name']
                    
                    # 제품명 변형 생성
                    self.add_product_variations(base_name, components['ingredient_kr'])
                
                # 성분명 기반 유의어
                if components['ingredient_kr'] and pd.notna(eng_ingredient):
                    self.link_ingredients(components['ingredient_kr'], eng_ingredient, atc_name)
    
    def add_product_variations(self, product_name, ingredient_kr=None):
        """제품명의 다양한 변형 생성"""
        variations = set()
        
        # 기본 제품명
        base = self.normalize_text(product_name)
        variations.add(base)
        
        # 일반적인 변형 패턴
        # 예: 로사탄아이 -> 로사탄
        if len(base) > 3:
            # 뒤에 붙은 회사 식별자 제거 (아이, 에스, 알 등)
            suffixes = ['아이', '에스', '알', '플러스', 'CR', 'SR', 'XR']
            for suffix in suffixes:
                if base.endswith(suffix):
                    variations.add(base[:-len(suffix)])
        
        # 성분명도 추가
        if ingredient_kr:
            ing_norm = self.normalize_text(ingredient_kr)
            variations.add(ing_norm)
            
            # 성분명의 일부 추출 (복합제의 경우)
            if '·' in ing_norm or ',' in ing_norm:
                parts = re.split('[·,]', ing_norm)
                for part in parts:
                    variations.add(part.strip())
        
        # 모든 변형을 서로 연결
        for var in variations:
            if var:  # 빈 문자열 제외
                self.synonym_dict[var].update(variations)
    
    def link_ingredients(self, kr_ingredient, en_ingredient, atc_name):
        """한글, 영문 성분명과 ATC 코드 연결"""
        synonyms = set()
        
        if pd.notna(kr_ingredient):
            kr_norm = self.normalize_text(kr_ingredient)
            synonyms.add(kr_norm)
            
            # 복합 성분 처리
            if '·' in kr_norm or ',' in kr_norm:
                parts = re.split('[·,]', kr_norm)
                for part in parts:
                    synonyms.add(part.strip())
        
        if pd.notna(en_ingredient):
            en_norm = en_ingredient.lower().strip()
            synonyms.add(en_norm)
            
            # 영문 성분명의 변형
            # 예: acetaminophen -> paracetamol
            en_variations = self.get_english_variations(en_norm)
            synonyms.update(en_variations)
        
        if pd.notna(atc_name):
            atc_norm = self.normalize_text(atc_name)
            synonyms.add(atc_norm)
        
        # 모든 유의어 연결
        for syn in synonyms:
            if syn:
                self.synonym_dict[syn].update(synonyms)
    
    def normalize_text(self, text):
        """텍스트 정규화"""
        if pd.isna(text):
            return ''
        
        # 소문자 변환, 공백 정리
        text = str(text).lower().strip()
        
        # 특수문자 정리
        text = re.sub(r'[^\w\s가-힣·,]', '', text)
        text = re.sub(r'\s+', ' ', text)
        
        return text
    
    def get_english_variations(self, eng_name):
        """영문 약품명의 일반적인 변형"""
        variations = set()
        
        # 일반적인 약물 이름 변형 매핑
        common_variations = {
            'acetaminophen': ['paracetamol', 'tylenol'],
            'amoxicillin': ['amoxil', 'augmentin'],
            'losartan': ['cozaar'],
            'atorvastatin': ['lipitor'],
            'omeprazole': ['prilosec'],
            'esomeprazole': ['nexium'],
            'metformin': ['glucophage'],
            'amlodipine': ['norvasc']
        }
        
        for key, values in common_variations.items():
            if key in eng_name.lower():
                variations.update(values)
        
        return variations
    
    def process_notification_patterns(self):
        """고시문서에 자주 나타나는 패턴 처리"""
        # 고시문서 특별 패턴
        notification_patterns = {
            # 생물학적 제제 그룹
            'tnf억제제': ['adalimumab', 'etanercept', 'golimumab', 'infliximab', 
                        '아달리무맙', '에타너셉트', '골리무맙', '인플릭시맙'],
            'jak억제제': ['baricitinib', 'tofacitinib', 'upadacitinib', 'filgotinib',
                        '바리시티닙', '토파시티닙', '유파다시티닙', '필고티닙',
                        '올루미언트', '젤잔즈'],
            'il6억제제': ['tocilizumab', '토실리주맙', '악템라'],
            
            # 약물 클래스별 그룹
            'dmards': ['methotrexate', 'mtx', '메토트렉세이트'],
            'nsaids': ['록소프로펜', 'loxoprofen', '아세클로페낙', 'aceclofenac'],
            'ppi': ['오메프라졸', 'omeprazole', '에스오메프라졸', 'esomeprazole'],
            
            # 제형 통일
            '경구제': ['정', '캡슐', 'tab', 'cap'],
            '주사제': ['주', 'inj', 'injection'],
            '외용제': ['겔', '크림', '연고', 'gel', 'cream', 'ointment']
        }
        
        for group_name, members in notification_patterns.items():
            for member in members:
                self.synonym_dict[member].add(group_name)
                self.synonym_dict[member].update(members)
    
    def generate_opensearch_config(self):
        """OpenSearch 설정 생성"""
        # 유의어 텍스트 파일 형식으로 변환
        synonym_lines = []
        
        # 중복 제거 및 정리
        processed_groups = set()
        for key, synonyms in self.synonym_dict.items():
            if synonyms:
                # 정렬하여 일관된 그룹 생성
                synonym_group = sorted(list(synonyms))
                group_key = ','.join(synonym_group)
                
                if group_key not in processed_groups:
                    processed_groups.add(group_key)
                    synonym_lines.append(','.join(synonym_group))
        
        # OpenSearch 설정
        opensearch_config = {
            "settings": {
                "index": {
                    "max_result_window": 50000,
                    "analysis": {
                        "tokenizer": {
                            "korean_tokenizer": {
                                "type": "nori_tokenizer",
                                "decompound_mode": "mixed",
                                "user_dictionary": "user_dictionary.txt"
                            }
                        },
                        "filter": {
                            "drug_synonym_filter": {
                                "type": "synonym_graph",
                                "synonyms": synonym_lines[:1000],  # 샘플로 1000개만
                                "updateable": True
                            },
                            "remove_dosage": {
                                "type": "pattern_replace",
                                "pattern": "\\d+(\\.\\d+)?\\s*(mg|밀리그램|g|그램|ml|밀리리터|%)",
                                "replacement": ""
                            },
                            "lowercase_filter": {
                                "type": "lowercase"
                            }
                        },
                        "analyzer": {
                            "drug_search_analyzer": {
                                "type": "custom",
                                "tokenizer": "korean_tokenizer",
                                "filter": [
                                    "lowercase_filter",
                                    "remove_dosage",
                                    "drug_synonym_filter",
                                    "nori_readingform"
                                ]
                            },
                            "drug_index_analyzer": {
                                "type": "custom",
                                "tokenizer": "korean_tokenizer",
                                "filter": [
                                    "lowercase_filter",
                                    "remove_dosage",
                                    "nori_readingform"
                                ]
                            }
                        }
                    }
                }
            },
            "mappings": {
                "properties": {
                    "drug_name": {
                        "type": "text",
                        "analyzer": "drug_index_analyzer",
                        "search_analyzer": "drug_search_analyzer",
                        "fields": {
                            "keyword": {
                                "type": "keyword",
                                "normalizer": "lowercase"
                            },
                            "ngram": {
                                "type": "text",
                                "analyzer": "ngram_analyzer"
                            }
                        }
                    },
                    "ingredient_kr": {
                        "type": "text",
                        "analyzer": "drug_search_analyzer"
                    },
                    "ingredient_en": {
                        "type": "text",
                        "analyzer": "standard"
                    },
                    "atc_code": {
                        "type": "keyword"
                    },
                    "content": {
                        "type": "text",
                        "analyzer": "korean_tokenizer"
                    }
                }
            }
        }
        
        return opensearch_config, synonym_lines
    
    def save_results(self, output_dir='./opensearch_config'):
        """결과 저장"""
        import os
        os.makedirs(output_dir, exist_ok=True)
        
        # OpenSearch 설정 저장
        config, synonyms = self.generate_opensearch_config()
        
        with open(f'{output_dir}/opensearch_settings.json', 'w', encoding='utf-8') as f:
            json.dump(config, f, ensure_ascii=False, indent=2)
        
        # 유의어 파일 저장
        with open(f'{output_dir}/drug_synonyms.txt', 'w', encoding='utf-8') as f:
            for line in synonyms:
                f.write(line + '\n')
        
        # 유의어 사전 저장 (디버깅용)
        with open(f'{output_dir}/synonym_dict.json', 'w', encoding='utf-8') as f:
            # set을 list로 변환
            dict_for_json = {k: list(v) for k, v in self.synonym_dict.items()}
            json.dump(dict_for_json, f, ensure_ascii=False, indent=2)
        
        print(f"총 {len(self.synonym_dict)} 개의 유의어 그룹 생성")
        return config

# 사용 예시
def main():
    # CSV 파일 읽기
    # df = pd.read_csv('drug_meta_master_v3_ko_20250825.csv')
    df = pd.read_csv('C:\Jimin\cg_suri_z-code360\pharmaLex_unity\result\drug_meta_master_v3_ko_20250825 (2).csv')
    
    # 유의어 생성기 초기화
    generator = DrugSynonymGenerator()
    
    # CSV 데이터 처리
    generator.process_csv_data(df)
    
    # 고시문서 패턴 추가
    generator.process_notification_patterns()
    
    # 결과 저장
    generator.save_results()
    
    # 테스트 쿼리
    test_queries = [
        "올루미언트",
        "baricitinib",
        "바리시티닙",
        "JAK 억제제",
        "로사탄",
        "아목시실린"
    ]
    
    print("\n=== 유의어 매칭 테스트 ===")
    for query in test_queries:
        normalized = generator.normalize_text(query)
        if normalized in generator.synonym_dict:
            print(f"{query} -> {list(generator.synonym_dict[normalized])[:5]}...")



In [6]:
if __name__ == "__main__":
    main()

OSError: [Errno 22] Invalid argument: 'C:\\Jimin\\cg_suri_z-code360\\pharmaLex_unity\result\\drug_meta_master_v3_ko_20250825 (2).csv'